In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import struct

import matplotlib.pyplot as plt

from spikeinterface.core import NumpySorting
import spikeinterface.extractors as se
from spikeinterface import create_sorting_analyzer


output_root = Path("/media/Projects/alana/UnitRefine/analyzers")

output_root.mkdir(parents=True, exist_ok=True)

print("Analyzer output root:")
print(output_root)


/opt/miniconda3/envs/unitrefine/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
metric_names = [
    "num_spikes",
    "firing_rate",
    "presence_ratio",
    "snr",
    "isi_violation",
    "rp_violation",
    "sliding_rp_violation",
    "synchrony",
    "firing_range",
    "sd_ratio",
    "amplitude_cutoff",
    "amplitude_median",
    "amplitude_cv",
    "drift"
]

template_metric_names = [
    "peak_to_trough_duration",
    "waveform_ratios",
    "half_width",
    "repolarization_slope",
    "recovery_slope",
    "waveform_baseline_flatness",
    # "peak_after_to_trough_ratio", # removed, should be included in waveform_ratios?
    "number_of_peaks",
    "velocity_fits",
    "exp_decay",
    "spread"
]

In [ ]:
root_dir = Path("/media/Neuralynx")
csv = pd.read_csv("/media/Projects/alana/UnitRefine/sessionCherryCounts.csv")

for session_name in csv["sessionname"]:
    print(session_name)

    session_path = root_dir / session_name

    df_channel_names = pd.read_csv(session_path / "ChannelNames.txt", header=None)
    df_channel_names["ch_name"] = df_channel_names[0].str.removesuffix(".ncs")

    recording = se.read_neuralynx(session_path)

    # get the first timestamp from a raw ncs file to use to align the spike times
    # necessary for later combing the sorting object with the analyzer
    ncs_path = session_path / "CSC1.ncs" 

    with open(ncs_path, "rb") as f:
        f.seek(16384)  
        first_record = f.read(1044)

    timestamp_us = struct.unpack("<Q", first_record[:8])[0] 
    print("First sample timestamp (µs since acquisition start):", timestamp_us)

    sampling_frequency = 32768
    offset_samples = round(timestamp_us * sampling_frequency / 1e6)
    print(offset_samples)

    channel_ids = np.arange(df_channel_names.shape[0]) + 1

    # channel names are taken from the recording system schema and don't match the auto reindexing that combinato does
    # create a lookup table here to skirt the issue

    ch_lookup = pd.DataFrame({
        "obj_name": recording.channel_ids,
        "ch_location": recording._properties["channel_name"]
    })

    ch_lookup["csc_nr"] = [np.int32(df_channel_names[df_channel_names["ch_name"] == row.ch_location].index)[0] + 1 for i, row in ch_lookup.iterrows()]

    for ch_nr in channel_ids:

        print("channel: ", ch_nr)
        path_file = session_path / f"CSC{ch_nr}"

        sorting = se.read_combinato(
            folder_path=path_file,
            sampling_frequency=32768, 
            user="tho",
            det_sign="pos",
            keep_good_only=False,
        )

        shifted_spike_trains = {
            unit_id: sorting.get_unit_spike_train(unit_id) - offset_samples
            for unit_id in sorting.unit_ids
        }

        sorting_shifted = NumpySorting.from_unit_dict(shifted_spike_trains, sampling_frequency=sorting.sampling_frequency)
        sorting_shifted.count_num_spikes_per_unit()

        ch_id = ch_lookup[ch_lookup["csc_nr"] == ch_nr]["obj_name"].iloc[0]
        recording_csc = recording.select_channels([ch_id])

        
        recording_csc.set_dummy_probe_from_locations(locations=np.array([[0, 0]]))

        analyzer = create_sorting_analyzer(
            sorting=sorting_shifted,
            recording=recording_csc,
            format="binary_folder",
            folder="analyzer_CSC1",
            sparse=False,
            overwrite=True
        )

        analyzer.compute("random_spikes")
        analyzer.compute("waveforms")
        analyzer.compute("templates")
        analyzer.compute("quality_metrics")
        analyzer.compute("noise_levels")
        analyzer.compute("correlograms")
        analyzer.compute("unit_locations")
        analyzer.compute("spike_amplitudes")
        analyzer.compute("template_similarity")
        analyzer.compute("spike_locations")
        analyzer.compute("template_metrics", metric_names=template_metric_names)
        analyzer.compute("quality_metrics", metric_names=metric_names)

    break

030e03ospr1-2014-05-02_10-45-58
First sample timestamp (µs since acquisition start): 158833902332
5204669312
